# Random forest — perinatal_anc cohort

Train **random forest** on the `perinatal_anc` cohort, evaluate on the held-out
test split, and record the result.

Owner: _(Dennis)_  ·  **Set `random_state=42` so results reproduce.**



## 1. Load the cohort

Load the cohort through the project's shared dataset builder. Retain the predefined training and held-out test assignments.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

current = Path.cwd().resolve()
project_root = next(
    (
        path for path in [current, *current.parents]
        if (path / "ml" / "src" / "dataset.py").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing ml/src/dataset.py"
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ml.src.dataset import build_cohort, cv_folds

COHORT = "perinatal_anc"
TARGET = "perinatal_death"
RANDOM_STATE = 42

df = build_cohort(COHORT)

train = df.loc[df["split"] == "train"].reset_index(drop=True)
test = df.loc[df["split"] == "test"].reset_index(drop=True)

print(f"Cohort shape: {df.shape}")
print(f"Training rows: {len(train):,} | Test rows: {len(test):,}")
print(
    f"Training events: {int(train[TARGET].sum())} | "
    f"Test events: {int(test[TARGET].sum())}"
)


## 2. Review the cohort

Review the target distribution, event prevalence, missing values and separation of clusters between the training and test sets. These checks help identify data issues before modelling.


In [ ]:
print("Target distribution:")
print(df[TARGET].value_counts(dropna=False).sort_index())
print(f"Overall event rate: {df[TARGET].mean():.2%}")

print("\\nMissing values:")
missing = df.isna().sum()
print(missing[missing > 0].sort_values(ascending=False))

train_clusters = set(train["cluster_id"])
test_clusters = set(test["cluster_id"])
cluster_overlap = train_clusters.intersection(test_clusters)

print(f"\\nTrain/test cluster overlap: {len(cluster_overlap)}")
if cluster_overlap:
    raise ValueError("Cluster overlap detected between training and test sets.")


## 3. Define predictors and preprocessing

Use the 11 agreed maternal and antenatal-care predictors. Numeric and binary variables are imputed within the modelling pipeline so that imputation is learned from each training fold and does not leak information from validation or test observations.


In [ ]:
NUMERIC_FEATURES = [
    "maternal_age",
    "prior_live_births",
    "preceding_interval_m",
]

BINARY_FEATURES = [
    "first_pregnancy",
    "prior_stillbirth",
    "prior_loss",
    "multiple",
    "no_anc",
    "anc4plus",
    "urine_tested",
    "iron_tablets",
]

FEATURES = NUMERIC_FEATURES + BINARY_FEATURES

X_train = train[FEATURES].copy()
X_test = test[FEATURES].copy()

y_train = train[TARGET].astype(int).copy()
y_test = test[TARGET].astype(int).copy()

print(f"Number of predictors: {len(FEATURES)}")
print("Predictors:", FEATURES)
print(f"Training matrix: {X_train.shape}")
print(f"Test matrix: {X_test.shape}")


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.base import clone


def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                SimpleImputer(strategy="median"),
                NUMERIC_FEATURES,
            ),
            (
                "binary",
                SimpleImputer(strategy="most_frequent"),
                BINARY_FEATURES,
            ),
        ],
        remainder="drop",
    )


def make_model(params):
    return Pipeline(
        steps=[
            ("preprocessor", make_preprocessor()),
            (
                "classifier",
                RandomForestClassifier(
                    **params,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ]
    )


## 4. Define Random Forest candidates

Evaluate the baseline and three constrained configurations. Class weighting is used to account for the low prevalence of the positive outcome. The same configurations are assessed using identical cross-validation folds.


In [ ]:
CONFIGURATIONS = {
    "Baseline": {
        "n_estimators": 400,
        "max_depth": None,
        "min_samples_leaf": 1,
    },
    "Model_A": {
        "n_estimators": 400,
        "max_depth": 8,
        "min_samples_leaf": 10,
    },
    "Model_B": {
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_leaf": 20,
    },
    "Model_C": {
        "n_estimators": 600,
        "max_depth": 10,
        "min_samples_leaf": 10,
    },
}

for model_name, parameters in CONFIGURATIONS.items():
    print(model_name, parameters)


## 5. Cluster-aware cross-validation

Use the project's five cluster-grouped folds on the training cohort. Confirm that no cluster appears in both the training and validation portions of any fold.

Compare configurations using PR-AUC as the primary metric, with ROC-AUC and recall as supporting metrics. The held-out test set is not used for model selection.


In [ ]:
from sklearn.model_selection import cross_validate

folds = cv_folds(train=train, name=COHORT, n_splits=5)

print(f"Number of folds: {len(folds)}")

for fold_number, (fit_indices, validation_indices) in enumerate(folds, start=1):
    fit_clusters = set(train.iloc[fit_indices]["cluster_id"])
    validation_clusters = set(train.iloc[validation_indices]["cluster_id"])
    overlap = fit_clusters.intersection(validation_clusters)

    print(
        f"Fold {fold_number}: "
        f"train={len(fit_indices):,}, "
        f"validation={len(validation_indices):,}, "
        f"cluster overlap={len(overlap)}"
    )

    if overlap:
        raise ValueError(
            f"Cluster leakage detected in fold {fold_number}."
        )

scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
}

cv_rows = []

for model_name, parameters in CONFIGURATIONS.items():
    candidate_model = make_model(parameters)

    scores = cross_validate(
        candidate_model,
        X_train,
        y_train,
        cv=folds,
        scoring=scoring,
        n_jobs=-1,
        error_score="raise",
    )

    row = {"model": model_name}

    for metric_name in scoring:
        values = scores[f"test_{metric_name}"]
        row[f"{metric_name}_mean"] = values.mean()
        row[f"{metric_name}_std"] = values.std()

    cv_rows.append(row)

cv_results = (
    pd.DataFrame(cv_rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

display(cv_results.round(4))


## 6. Select and refit the model

Select the configuration with the highest mean cross-validation PR-AUC. Review recall and fold-to-fold variability alongside the primary metric.

After selection, refit the chosen pipeline on the complete training set. The held-out test set remains untouched until the final evaluation.


In [ ]:
best_row = cv_results.iloc[0]

selected_model_name = best_row["model"]
selected_parameters = CONFIGURATIONS[selected_model_name]

print("Selected configuration:", selected_model_name)
print(
    f"Mean CV PR-AUC: {best_row['pr_auc_mean']:.4f} "
    f"± {best_row['pr_auc_std']:.4f}"
)
print(
    f"Mean CV ROC-AUC: {best_row['roc_auc_mean']:.4f} "
    f"± {best_row['roc_auc_std']:.4f}"
)
print(
    f"Mean CV recall: {best_row['recall_mean']:.4f} "
    f"± {best_row['recall_std']:.4f}"
)

final_model = make_model(selected_parameters)
final_model.fit(X_train, y_train)

print("\\nSelected model refitted on the complete training set.")


## 7. Final held-out test evaluation

Evaluate the selected model once on the untouched test set. Report PR-AUC as the primary metric, alongside ROC-AUC and the constant baseline (the event rate), pooled and for Kenya and Tanzania. Country is not a model feature; it is used here only to split the results.


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

y_test_prob = final_model.predict_proba(X_test)[:, 1]

test_pr_auc = average_precision_score(y_test, y_test_prob)
test_roc_auc = roc_auc_score(y_test, y_test_prob)


def test_metrics(mask):
    y, p = y_test[mask], y_test_prob[mask]
    return {
        "rows": int(mask.sum()),
        "events": int(y.sum()),
        "baseline_pr_auc": y.mean(),
        "pr_auc": average_precision_score(y, p),
        "roc_auc": roc_auc_score(y, p),
    }


# pooled and per country (country is not a feature)
test_scores = pd.DataFrame({
    "pooled": test_metrics(np.ones(len(test), bool)),
    "Kenya": test_metrics((test["country"] == "Kenya").to_numpy()),
    "Tanzania": test_metrics((test["country"] == "Tanzania").to_numpy()),
}).T

print("Final test metrics:")
print(test_scores.round(4))
print()
print(f"Mean predicted risk: {y_test_prob.mean():.3f} | observed rate: {y_test.mean():.3f}")

### Test Results

The selected configuration (Model_B: 400 trees, maximum depth 12, at least 20 observations per leaf) achieved a PR-AUC of 0.0887 on 3,310 test pregnancies with 108 perinatal deaths, against a constant baseline of 0.0326 (the event rate). This is about 2.7 times the baseline. Its ROC-AUC of 0.6280 indicates modest discriminatory ability.

Performance is stronger in Kenya (PR-AUC 0.1115, ROC-AUC 0.6583; baseline 0.0305) than in Tanzania (PR-AUC 0.0744, ROC-AUC 0.5905; baseline 0.0361).

The mean predicted risk is 0.364 against an observed rate of 0.033, because balanced class weights inflate the probabilities. The scores should be used to rank pregnancies, not read as risks.

## 8. Compare with the clinical checklist

Apply the agreed checklist to the held-out test set. The model flags the same number of pregnancies as the checklist (its highest scores), so both approaches are compared at the same screening capacity: events caught, sensitivity and PPV.


In [ ]:
# checklist flag: same rule as checklist_flag in dataset.py
checklist_flag = (
    (test["maternal_age"] < 18)
    | (test["maternal_age"] >= 35)
    | (test["prior_live_births"] >= 5)
    | (test["preceding_interval_m"] < 24)
    | (test["multiple"] == 1)
    | (test["prior_stillbirth"] == 1)
).to_numpy()
n_flagged = int(checklist_flag.sum())

# model flags the same number of pregnancies: the highest scores
model_flag = np.zeros(len(test), bool)
model_flag[np.argsort(-y_test_prob, kind="stable")[:n_flagged]] = True

# deaths caught, sensitivity and PPV at the same flag count
events = y_test.to_numpy() == 1
compare = pd.DataFrame({
    "flagged": [n_flagged, n_flagged],
    "events_caught": [int((checklist_flag & events).sum()), int((model_flag & events).sum())],
}, index=["checklist", "random forest"])
compare["sensitivity"] = compare["events_caught"] / events.sum()
compare["ppv"] = compare["events_caught"] / compare["flagged"]

print(f"Checklist flag rate: {checklist_flag.mean():.1%} ({n_flagged:,} pregnancies)")
print(compare.round(3))

# rows can share a score at the cut-off: range of deaths the model could catch
cut = np.sort(y_test_prob)[::-1][n_flagged - 1]
above, tied = y_test_prob > cut, y_test_prob == cut
slots = n_flagged - int(above.sum())
low = int((above & events).sum()) + max(0, slots - int((tied & ~events).sum()))
high = int((above & events).sum()) + min(slots, int((tied & events).sum()))
print(f"{int(tied.sum())} rows tied at the cut-off; model catches {low}-{high} deaths")

recall_at_flag = float(compare.loc["random forest", "sensitivity"])
beats_checklist = bool(compare.loc["random forest", "events_caught"]
                       > compare.loc["checklist", "events_caught"])
print(f"Beats checklist: {beats_checklist}")

### Key Findings

The checklist flags 1,457 test pregnancies (44.0%). Flagging the same 1,457 pregnancies by model score, the Random Forest model identified 66 of the 108 observed perinatal deaths, compared with 49 identified by the existing checklist.

This represents:

17 additional perinatal death cases identified.

A 15.74 percentage-point improvement in recall (sensitivity 61.11% vs 45.37%).

A positive predictive value of 4.5%, against 3.4% for the checklist.

Approximately 34.7% more deaths identified than the checklist, at the same flagging volume.

### Conclusion

The Random Forest model outperformed the existing checklist in identifying perinatal death cases when both approaches were evaluated at an equal flagging volume. Its recall of 61.11%, compared with 45.37% for the checklist, demonstrates improved case identification without increasing the number of patients flagged.

These findings support further development and validation of the Random Forest model as a potential risk-screening support tool. However, its false-positive rate, clinical suitability and performance in independent populations should also be assessed before practical deployment.

## 9. Save evaluation metrics

Save the selected model's held-out test metrics to the shared application database. Keep `is_best=False` until the team has compared all algorithms and formally selected the deployment model.



In [ ]:
# Random Forest results for the shared hosted metrics database
row = {
    "cohort": COHORT,
    "algorithm": "random_forest",
    "pr_auc": float(test_pr_auc),
    "roc_auc": float(test_roc_auc),
    "recall_at_flag": float(recall_at_flag),  # sensitivity at the checklist's flag count
    "beats_checklist": bool(beats_checklist),
    "test_events": int(y_test.sum()),
    "is_best": False,  # Pending overall model comparison
}

print("Metrics prepared for submission:")
print(pd.DataFrame([row]).T)

# Change to True when ready to submit the results.
SAVE_METRICS = False

if SAVE_METRICS:
    from db.metrics import save_metrics
    save_metrics(**row)